# 03 - Model Interpretation with SHAP

## AI-Generated Text Detection Project

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Models and Data

In [ ]:
model = joblib.load('../models/best_model.pkl')
preprocessing = joblib.load('../models/preprocessing.pkl')

tfidf = preprocessing['word_tfidf']
char_tfidf = preprocessing['char_tfidf']

test_df = pd.read_csv('../data/processed/test.csv')
X_test = test_df['text']
y_test = test_df['label']

print('Model and data loaded successfully')

## 2. Transform Test Data

In [ ]:
from scipy.sparse import hstack

X_test_word = tfidf.transform(X_test.astype(str))
X_test_char = char_tfidf.transform(X_test.astype(str))
X_test_combined = hstack([X_test_word, X_test_char])

X_test_dense = X_test_combined[:1000].toarray()

print(f'Test data shape (sample): {X_test_dense.shape}')

## 3. SHAP Analysis

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_dense)

## 4. Feature Names

In [ ]:
word_features = tfidf.get_feature_names_out()
char_features = char_tfidf.get_feature_names_out()
all_features = list(word_features) + list(char_features)

print(f'Total features: {len(all_features)}')

## 5. SHAP Summary Plot

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_dense, feature_names=all_features, max_display=20, show=False)
plt.tight_layout()
plt.savefig('../app/assets/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Top Features for AI Detection

In [ ]:
mean_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({'feature': all_features[:len(mean_shap)], 'importance': mean_shap})
feature_importance = feature_importance.sort_values('importance', ascending=False).head(30)

print('Top 30 Most Important Features:')
print(feature_importance)

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(20), x='importance', y='feature', palette='viridis')
plt.title('Top 20 Features by SHAP Importance')
plt.xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.savefig('../app/assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. SHAP Dependence Plot

In [ ]:
top_feature_idx = feature_importance.iloc[0]['feature']
if top_feature_idx in all_features:
    feature_idx = all_features.index(top_feature_idx)
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(feature_idx, shap_values, X_test_dense, feature_names=all_features, show=False)
    plt.tight_layout()
    plt.savefig('../app/assets/shap_dependence.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Save Interpretation Results

In [ ]:
feature_importance.to_csv('../reports/feature_importance.csv', index=False)
print('Feature importance saved to reports/')

## 9. Key Insights

In [ ]:
print('='*60)
print('KEY INSIGHTS FROM SHAP ANALYSIS')
print('='*60)
print('\nTop 10 features most indicative of AI-generated text:')
for i, row in feature_importance.head(10).iterrows():
    print(f"  - {row['feature']}: {row['importance']:.4f}")
print('\nThese features help distinguish AI vs Human text patterns.')
print('='*60)